# Match Customer Records Across Systems, Then Pick The Threshold

Companion notebook for the [Cleaning and Entity Resolution](https://codemia.io/courses/forward_deployed_engineering/fde_cleaning_and_entity_resolution) lesson on Codemia.

**What you will build**: a small entity resolution pipeline of the kind an FDE writes in the first
weeks of a deployment, when the customer has the same companies in a CRM and a billing system and
nobody trusts either. You will generate two noisy record sets with a known truth, normalise names
and addresses, block candidates so the comparison stays cheap, score pairs with a weighted
similarity and then choose a match threshold by looking at precision, recall and the cost of each
kind of mistake. You will finish with the review queue a customer analyst would actually work.

No GPU. Standard library, `numpy` and `matplotlib` only.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import re
import random
import difflib
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

random.seed(7)
np.random.seed(7)

## Concept recap

Entity resolution decides which records describe the same real thing. In the field it is rarely
glamorous: one system says `Kestrel Corp.`, another says `KESTREL CORPORATION LTD`, a third has a typo
in the postcode. Comparing every record with every other is quadratic, so pipelines first
**block** records into small candidate groups using a cheap key, then **score** only the pairs
inside each block, then **decide** with a threshold. Pairs near the threshold go to a human.

The threshold is a business decision as much as a technical one. A false merge can send one
customer's invoice to another company; a missed match only leaves a duplicate. The lesson called
this agreeing the cost of each error with the customer before tuning anything.

## Generate two systems with a known truth

In [ ]:
BASE = ['brightfield', 'harrow', 'kestrel', 'lumen', 'marlow', 'northgate', 'oakridge', 'pellam',
        'quarry hill', 'redwater', 'saltmarsh', 'tidewell', 'upton', 'vantage', 'westbrook', 'yarrow',
        'ashcombe', 'bellmont', 'corran', 'dunmore', 'elmstead', 'fernway', 'glenrock', 'holloway']
SUFFIX = ['inc', 'corp', 'corporation', 'ltd', 'llc', 'group', 'holdings', '']
STREETS = ['market st', 'high st', 'main st', 'king rd', 'park ave', 'river rd', 'mill lane']
CITIES = ['london', 'leeds', 'bristol', 'york', 'bath', 'derby']

def typo(s, p=0.08):
    out = []
    for ch in s:
        r = random.random()
        if r < p / 3 and ch.isalpha():
            continue                                   # drop a letter
        elif r < 2 * p / 3 and ch.isalpha():
            out.append(random.choice('abcdefghijklmnopqrstuvwxyz'))  # wrong letter
        else:
            out.append(ch)
    return ''.join(out)

entities = []
for i in range(400):
    stem = random.choice(BASE) + ('' if random.random() < 0.5 else ' ' + random.choice(['labs', 'energy', 'foods', 'systems', 'bank']))
    entities.append({'id': i,
                     'name': stem + ' ' + random.choice(SUFFIX),
                     'street': f'{random.randint(1, 300)} {random.choice(STREETS)}',
                     'city': random.choice(CITIES),
                     'postcode': f'{random.choice("ABCDEFGH")}{random.randint(1, 20)} {random.randint(1, 9)}{random.choice("XYZ")}'})

# Branches: a second legal entity with the same name in the same city, a few doors away and in
# the same postcode district. These are the pairs that tempt a matcher into a false merge.
for e in random.sample(entities, 50):
    sib = dict(e, id=len(entities))
    sib['street'] = f"{int(e['street'].split()[0]) + random.randint(2, 40)} {e['street'].split(' ', 1)[1]}"
    sib['postcode'] = e['postcode'][:-2] + str(random.randint(1, 9)) + random.choice('XYZ')
    entities.append(sib)

def render_crm(e):
    return {'src': 'crm', 'truth': e['id'], 'name': e['name'].title() + random.choice(['', '.', ',']),
            'address': f"{e['street'].title()}, {e['city'].title()}", 'postcode': e['postcode']}

def render_billing(e):
    name = typo(e['name'].upper())
    if random.random() < 0.3:
        name = name.replace('CORPORATION', 'CORP').replace('LIMITED', 'LTD')
    pc = e['postcode'].replace(' ', '') if random.random() < 0.5 else e['postcode']
    if random.random() < 0.05:
        pc = pc[:-1] + random.choice('XYZ')            # a wrong last character
    if random.random() < 0.06:
        pc = ''                                        # never captured in billing
    return {'src': 'billing', 'truth': e['id'], 'name': name,
            'address': typo(f"{e['street']} {e['city']}".upper(), 0.04), 'postcode': pc}

# Not every customer is in both systems, which is normal in the field.
crm = [render_crm(e) for e in entities if random.random() < 0.9]
billing = [render_billing(e) for e in entities if random.random() < 0.85]
true_pairs = {(i, j) for i, a in enumerate(crm) for j, b in enumerate(billing) if a['truth'] == b['truth']}
print(len(crm), 'crm records,', len(billing), 'billing records,', len(true_pairs), 'true matches')
for r in (crm[0], billing[0]):
    print(r)

Names in the two systems share a stem but differ in case, punctuation, legal suffix and typos.
Some stems repeat across different companies, which is what makes naive matching on the name
alone dangerous: two `Kestrel Labs` branches in the same city are different legal entities and must stay separate.

## Normalise

In [ ]:
LEGAL = {'inc', 'corp', 'corporation', 'ltd', 'limited', 'llc', 'group', 'holdings', 'co'}

def norm_name(s):
    s = re.sub(r'[^a-z0-9 ]', ' ', s.lower())
    return ' '.join(t for t in s.split() if t not in LEGAL)

def norm_pc(s):
    return re.sub(r'\s', '', s.upper())

def norm_addr(s):
    s = re.sub(r'[^a-z0-9 ]', ' ', s.lower())
    s = re.sub(r'\bstreet\b', 'st', s)
    s = re.sub(r'\broad\b', 'rd', s)
    return ' '.join(s.split())

for r in crm + billing:
    r['n_name'], r['n_pc'], r['n_addr'] = norm_name(r['name']), norm_pc(r['postcode']), norm_addr(r['address'])

print(crm[0]['name'], '->', crm[0]['n_name'])
print(billing[0]['name'], '->', billing[0]['n_name'])

## Block, so you do not compare everything with everything

A blocking key must be cheap and must put true matches in the same block. Here we try two keys:
the postcode district (the part before the digit run ends) and the first three letters of the
normalised name. A pair becomes a candidate if it shares either key. Watch two numbers: how many
pairs we must score, and how many true matches blocking already lost.

In [ ]:
def district(pc):
    m = re.match(r'[A-Z]+\d+', pc)
    return m.group(0) if m else None

def nkey(v):
    return v if v else object()                        # a missing key never shares a block

def block(records, keyfns):
    idx = defaultdict(set)
    for i, r in enumerate(records):
        for name, fn in keyfns.items():
            idx[(name, nkey(fn(r)))].add(i)
    return idx

keyfns = {'district': lambda r: district(r['n_pc']), 'name3': lambda r: r['n_name'][:3]}
for chosen in (['district'], ['name3'], ['district', 'name3']):
    fns = {k: keyfns[k] for k in chosen}
    a_idx, b_idx = block(crm, fns), block(billing, fns)
    cands = set()
    for key, ai in a_idx.items():
        for i in ai:
            for j in b_idx.get(key, ()):
                cands.add((i, j))
    kept = len(cands & true_pairs) / len(true_pairs)
    print(f'{"+".join(chosen):<15} candidates {len(cands):>6} of {len(crm) * len(billing):>6}   true matches kept {kept:.1%}')
candidates = cands

Either key alone loses some true matches: a typo in the first letters of a name, or a wrong
postcode, puts the pair in different blocks. The union keeps nearly all of them while still
scoring a small fraction of the full cross product. Blocking recall is a ceiling: no threshold
later can find a match that blocking threw away, so measure it first.

## Score candidate pairs

In [ ]:
def sim(a, b):
    return difflib.SequenceMatcher(None, a, b).ratio()

def score(a, b):
    name = sim(a['n_name'], b['n_name'])
    addr = sim(a['n_addr'], b['n_addr'])
    if not a['n_pc'] or not b['n_pc']:
        pc = 0.5                                       # unknown, neither evidence for nor against
    else:
        pc = 1.0 if a['n_pc'] == b['n_pc'] else (0.5 if district(a['n_pc']) == district(b['n_pc']) else 0.0)
    return 0.5 * name + 0.3 * addr + 0.2 * pc

pairs = sorted(candidates)
scores = np.array([score(crm[i], billing[j]) for i, j in pairs])
is_true = np.array([(i, j) in true_pairs for i, j in pairs])

fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.linspace(0, 1, 41)
ax.hist(scores[~is_true], bins=bins, alpha=0.6, label='different companies')
ax.hist(scores[is_true], bins=bins, alpha=0.6, label='same company')
ax.set_yscale('log')
ax.set_xlabel('pair score')
ax.set_ylabel('candidate pairs (log)')
ax.legend()
ax.set_title('Matches and non matches overlap in the middle')
plt.tight_layout()
plt.show()

The two distributions separate well at the ends and overlap in the middle. That overlap is where
every hard decision lives: a branch with the same name a few doors away, or the same company with
a mangled name and no postcode.

## Choose a threshold with the customer's costs

In [ ]:
def at(t):
    pred = scores >= t
    tp = int((pred & is_true).sum())
    fp = int((pred & ~is_true).sum())
    fn = len(true_pairs) - tp                      # includes matches lost by blocking
    return tp, fp, fn

ts = np.linspace(0.4, 0.98, 59)
prec, rec = [], []
for t in ts:
    tp, fp, fn = at(t)
    prec.append(tp / max(tp + fp, 1))
    rec.append(tp / (tp + fn))

# Agreed with the customer: a false merge costs 20 units of analyst clean up and risk,
# a missed match costs 1 unit (a duplicate stays).
COST_FP, COST_FN = 20, 1
cost = [COST_FP * at(t)[1] + COST_FN * at(t)[2] for t in ts]
best = ts[int(np.argmin(cost))]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.plot(ts, prec, label='precision')
a1.plot(ts, rec, label='recall')
a1.axvline(best, ls='--', color='gray')
a1.set_xlabel('threshold'); a1.legend(); a1.set_title('Precision and recall')
a2.plot(ts, cost)
a2.axvline(best, ls='--', color='gray')
a2.set_xlabel('threshold'); a2.set_ylabel('total cost'); a2.set_title(f'Cost minimised at {best:.2f}')
plt.tight_layout()
plt.show()
tp, fp, fn = at(best)
print(f'threshold {best:.2f}: {tp} matched, {fp} false merges, {fn} missed')

print('\nhow the best threshold moves with the cost of a false merge:')
for c_fp in (1, 5, 20, 100):
    c = [c_fp * at(t)[1] + COST_FN * at(t)[2] for t in ts]
    t = ts[int(np.argmin(c))]
    tp, fp, fn = at(t)
    print(f'  false merge costs {c_fp:>3}x a miss -> threshold {t:.2f}  ({fp} false merges, {fn} missed)')

When a false merge is cheap the best threshold drops and more pairs are merged; when it is
expensive the threshold rises and more duplicates are left alone. The same data gives a different answer when the business priorities change, which is
why the threshold belongs in the agreed acceptance criteria and not in a constant someone picked.

## The review band

Rather than one threshold, production pipelines usually use two: auto match above the high one,
auto reject below the low one, and send the band between them to a person. The question for the
customer becomes how many reviews per week their analysts can absorb.

In [ ]:
def band(lo, hi):
    auto = scores >= hi
    review = (scores >= lo) & (scores < hi)
    fp_auto = int((auto & ~is_true).sum())
    tp_total = int(((auto | review) & is_true).sum())   # assume reviewers decide correctly
    return int(review.sum()), fp_auto, tp_total / len(true_pairs)

print(f'{"low":>5} {"high":>5} {"to review":>10} {"auto false merges":>18} {"recall":>7}')
for lo, hi in ((0.84, 0.88), (0.80, 0.90), (0.75, 0.92), (0.70, 0.95)):
    n, fpa, r = band(lo, hi)
    print(f'{lo:>5.2f} {hi:>5.2f} {n:>10} {fpa:>18} {r:>7.1%}')

lo, hi = 0.80, 0.90
queue = [(scores[k], is_true[k], crm[i]['name'], billing[j]['name'], crm[i]['address'], billing[j]['address'])
         for k, (i, j) in enumerate(pairs) if lo <= scores[k] < hi]
random.shuffle(queue)
print('\nsample of the review queue an analyst would see (truth shown for you, not for them):')
for sc, t, a, b, aa, ab in sorted(queue[:10], reverse=True):
    print(f'  {sc:.2f} {"same" if t else "DIFF"}  {a:<26} | {b:<26} | {aa:<24} | {ab}')

Widening the band buys recall and removes automatic false merges, and it costs analyst time.
Show the customer this table; it turns an abstract precision number into a staffing decision
they can make.

Now read the queue itself. Most rows are the same company where a typo landed in the legal
suffix (`LrD`, `GROU`, `COPORATIO`), so normalisation failed to strip it and the name score
fell. Looking at the review queue is the fastest way to find pipeline bugs: fixing
normalisation shrinks the queue more than any threshold change will.

## Exercises

1. Add a third blocking key, such as the first four characters of the normalised street, and
   measure blocking recall and candidate count again. Is the extra key worth it?
2. Replace `difflib` with token set similarity on names (Jaccard on word sets). Which kind of
   error does it fix, and which does it introduce?
3. Make the generator create two different companies with the same stem in the same postcode.
   How does the scoring function need to change to keep them apart?
4. Make `norm_name` also drop any token within one edit of a legal suffix. How much smaller is
   the review band at the same recall?
5. Enforce one to one matching: each CRM record may match at most one billing record. Keep the
   highest scoring pair and measure precision again.

## What to take away

- Normalise before comparing: case, punctuation and legal suffixes cause most surface mismatch.
- Measure blocking recall first. It caps everything that follows.
- A weighted score over several fields separates matches far better than the name alone.
- Choose the threshold from the customer's costs of a false merge and a missed match.
- Use a review band for the uncertain middle, and size it with the customer's analyst capacity.